# Chapter 4: Reading and Writing Data with Pandas

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [1]:
import pandas as pd
import numpy as np

# Reading and Writing Data with Pandas

Pandas provides a comprehensive set of tools for importing and exporting data across multiple formats. Whether you're working with CSV files, Excel spreadsheets, databases, or specialized data formats, pandas offers a consistent and intuitive API for data I/O operations.

## Understanding the Pandas I/O API

The pandas I/O system follows a simple naming convention:
- **Reader functions**: `pd.read_*()` — used to import data
- **Writer methods**: `df.to_*()` — used to export data

This consistent pattern makes it easy to work with different formats without learning completely different syntax.

## Working with CSV Files

### Reading CSV Files

CSV (Comma-Separated Values) is one of the most common data formats. The `read_csv()` function is highly flexible and handles various CSV dialects.

In [9]:
import pandas as pd

# Basic CSV reading
df = pd.read_csv('data/people.csv')

# Reading with custom delimiter (tab-separated)
df = pd.read_csv('data/people.tsv', sep='\t')

# Reading without header row
df = pd.read_csv('data/people.csv', header=None)

# Specifying column names
df = pd.read_csv('data/people.csv', names=['Name', 'Age', 'City'])

# Reading only specific columns
df = pd.read_csv('data/people.csv', usecols=['Name', 'Age'])

# Handling missing values
df = pd.read_csv('data/people.csv', na_values=['N/A', 'NULL', ''])

# Reading large files in chunks
def process_data(chunk):
    print(chunk.shape)

chunks = pd.read_csv('data/large_file.csv', chunksize=1000)

for chunk in chunks:
    process_data(chunk)

### Or use a real operation directly:
chunks = pd.read_csv('data/large_file.csv', chunksize=1000)

for chunk in chunks:
    print(chunk['Age'].sum())

(1000, 3)
(1000, 3)
44500
44500


### Writing CSV Files

Exporting DataFrames to CSV format is straightforward with the `to_csv()` method.

In [10]:
import pandas as pd

# Basic CSV export
df.to_csv('output.csv')

# Exclude index column
df.to_csv('output.csv', index=False)

# Custom delimiter
df.to_csv('output.tsv', sep='\t', index=False)

# Specify which columns to write
df.to_csv('output.csv', columns=['Name', 'Age'], index=False)

# Custom missing value representation
df.to_csv('output.csv', na_rep='MISSING', index=False)

# Control floating-point precision
df.to_csv('output.csv', float_format='%.2f', index=False)

## Working with Excel Files

Excel files are widely used in business environments. Pandas supports both reading and writing Excel files through the `openpyxl` or `xlrd` libraries.

**Note:** Excel export requires `openpyxl`. Install if needed:
```bash
uv add openpyxl
pip install openpyxl
conda install openpyxl
```

### Reading Excel Files

In [13]:
import pandas as pd

# Read from default sheet
df = pd.read_excel('data/spreadsheet.xlsx')

# Read specific sheet by name
df = pd.read_excel('data/spreadsheet.xlsx', sheet_name='Sales')

# Read specific sheet by index (0-based)
df = pd.read_excel('data/spreadsheet.xlsx', sheet_name=0)

# Read multiple sheets into a dictionary
sheets = pd.read_excel('data/spreadsheet.xlsx', sheet_name=[0, 1, 'Summary'])

# Read with specific header row
df = pd.read_excel('data/spreadsheet.xlsx', header=1)

# Read specific rows and columns
df = pd.read_excel('data/spreadsheet.xlsx', usecols='A:C', nrows=100)

### Writing Excel Files

In [14]:
import pandas as pd

df = pd.DataFrame({
    'name': ['Alice', 'Bob'],
    'salary': [50000, 60000]
})

# Basic Excel export
df.to_excel('data/output.xlsx', index=False)

# Write to a specific sheet
df.to_excel('data/output.xlsx', sheet_name='Data', index=False)

# Create additional DataFrames
df1 = pd.DataFrame({'product': ['A', 'B'], 'sales': [100, 200]})
df2 = pd.DataFrame({'product': ['A', 'B'], 'stock': [50, 75]})
df3 = pd.DataFrame({'metric': ['Total Sales'], 'value': [300]})

# Write multiple DataFrames to different sheets
with pd.ExcelWriter('data/output.xlsx') as writer:
    df1.to_excel(writer, sheet_name='Sales', index=False)
    df2.to_excel(writer, sheet_name='Inventory', index=False)
    df3.to_excel(writer, sheet_name='Summary', index=False)

# Control worksheet placement
with pd.ExcelWriter('data/output.xlsx', engine='openpyxl') as writer:
    df.to_excel(
        writer,
        sheet_name='Data',
        startrow=2,
        startcol=1,
        index=False
    )

## Working with JSON Files

JSON (JavaScript Object Notation) is ideal for web applications and APIs. Pandas can read and write JSON data in multiple orientations.

### Reading JSON Files

In [19]:
import pandas as pd

# Default: records-oriented JSON
df = pd.read_json('data/data.json')

# If the JSON is a list of records:
# [
#   {"name": "Alice", "salary": 50000},
#   {"name": "Bob", "salary": 60000}
# ]
df = pd.read_json('data/data.json', orient='records')

# If the JSON is index-oriented:
# {
#   "0": {"name": "Alice", "salary": 50000},
#   "1": {"name": "Bob", "salary": 60000}
# }
df = pd.read_json('data/index_data.json', orient='index')

# If the JSON is column-oriented:
# {
#   "name": {"0": "Alice", "1": "Bob"},
#   "salary": {"0": 50000, "1": 60000}
# }
df = pd.read_json('data/column_data.json', orient='columns')

# If the JSON uses split orientation:
# {
#   "index": [0, 1],
#   "columns": ["name", "salary"],
#   "data": [["Alice", 50000], ["Bob", 60000]]
# }
df = pd.read_json('data/split_data.json', orient='split')

### Writing JSON Files

In [20]:
import pandas as pd

# Basic JSON export
df.to_json('output.json')

# Different output orientations
df.to_json('output.json', orient='records')  # Array of objects
df.to_json('output.json', orient='index')    # Object with index keys
df.to_json('output.json', orient='split')    # Structured format

# Pretty-print JSON
df.to_json('output.json', indent=2)

# Handle date formatting
df.to_json('output.json', date_format='iso', indent=2)

## Working with SQL Databases

Pandas integrates seamlessly with SQL databases, allowing you to read from and write to various database systems.

### Reading from SQL

In [22]:
import pandas as pd
import sqlalchemy
from pathlib import Path

# ----------------------------
# 1. Create sample data
# ----------------------------
df_users = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'email': ['a@x.com', 'b@x.com', 'c@x.com', 'd@x.com'],
    'age': [22, 35, 28, 40]
})

# ----------------------------
# 2. Create database in /data folder
# ----------------------------
Path("data").mkdir(exist_ok=True)
engine = sqlalchemy.create_engine('sqlite:///data/database.db')

# Write table to DB
df_users.to_sql('users', engine, if_exists='replace', index=False)

# ----------------------------
# 3. Read entire table
# ----------------------------
df = pd.read_sql_table('users', engine)
print(df)

# ----------------------------
# 4. Query results
# ----------------------------
query = "SELECT * FROM users WHERE age > 25"
df = pd.read_sql_query(query, engine)
print(df)

# ----------------------------
# 5. Select specific columns
# ----------------------------
df = pd.read_sql_query("SELECT name, email FROM users", engine)
print(df)

# ----------------------------
# 6. Chunked reading (large tables)
# ----------------------------
def process_data(chunk):
    print("Processing chunk:", len(chunk))

chunks = pd.read_sql_query(
    "SELECT * FROM users",
    engine,
    chunksize=2
)

for chunk in chunks:
    process_data(chunk)

   id     name    email  age
0   1    Alice  a@x.com   22
1   2      Bob  b@x.com   35
2   3  Charlie  c@x.com   28
3   4    Diana  d@x.com   40
   id     name    email  age
0   2      Bob  b@x.com   35
1   3  Charlie  c@x.com   28
2   4    Diana  d@x.com   40
      name    email
0    Alice  a@x.com
1      Bob  b@x.com
2  Charlie  c@x.com
3    Diana  d@x.com
Processing chunk: 2
Processing chunk: 2


### Writing to SQL

In [23]:
import pandas as pd
import sqlalchemy
from sqlalchemy import String, Integer, Float
from pathlib import Path

# ----------------------------
# Create sample DataFrame
# ----------------------------
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35],
    'salary': [50000, 60000, 70000]
})

# ----------------------------
# Create database in /data folder
# ----------------------------
Path("data").mkdir(exist_ok=True)
engine = sqlalchemy.create_engine('sqlite:///data/database.db')

# ----------------------------
# Write DataFrame to new table
# ----------------------------
df.to_sql('new_table', engine, if_exists='fail', index=False)

# ----------------------------
# Append to existing table
# ----------------------------
df.to_sql('existing_table', engine, if_exists='append', index=False)

# ----------------------------
# Replace existing table
# ----------------------------
df.to_sql('table_name', engine, if_exists='replace', index=False)

# ----------------------------
# Specify data types for columns
# ----------------------------
dtypes = {
    'name': String(50),
    'age': Integer(),
    'salary': Float()
}

df.to_sql(
    'employees',
    engine,
    dtype=dtypes,
    if_exists='replace',
    index=False
)

3

## Working with Parquet Files

Parquet is a columnar storage format that provides excellent compression and performance for large datasets. It also preserves data types across read/write cycles, unlike CSV.

In [4]:
import pandas as pd

# Read Parquet file (requires pyarrow or fastparquet)
df = pd.read_parquet('data/data.parquet')
print(df.head() )
# Write to Parquet
df.to_parquet('data/output.parquet', index=False)

# Specify compression
df.to_parquet(
    'data/output_compressed.parquet',
    compression='snappy',
    index=False
)

# Read specific columns (column pruning for efficiency)
df = pd.read_parquet(
    'data/data.parquet',
    columns=['Name', 'Age']
)

    Name  Age         City
0  Alice   30     New York
1    Bob   25  Los Angeles
2  Carol   38      Chicago
3   Dave   40      Houston
4    Eve   35      Seattle


## Working with HDF5 Files

HDF5 is a binary format ideal for storing large scientific datasets with hierarchical structure.

In [11]:
import pandas as pd

# -----------------------------
# 1. READ data from HDF5 file
# -----------------------------
df = pd.read_hdf('data/data.h5', key='another_key')

# Quick sanity check: preview data
print("=== RAW DATA (first 5 rows) ===")
print(df.head())
print("\nShape:", df.shape)
print("Columns:", df.columns.tolist())

# -----------------------------
# 2. WRITE data back to HDF5
#    (overwrite file safely)
# -----------------------------
df.to_hdf(
    'data/data.h5',
    key='another_key',
    format='table',   # required for filtering support
    mode='w'          # overwrite existing file
)

# -----------------------------
# 3. READ WITH FILTER (WHERE CLAUSE)
# -----------------------------
df_filtered = pd.read_hdf(
    'data/data.h5',
    key='another_key',
    where='index > 100'   # must be a STRING, not a list
)

# -----------------------------
# 4. OUTPUT CHECK FOR FILTERED DATA
# -----------------------------
print("\n=== FILTERED DATA (index > 100) ===")
print(df_filtered.head())
print("\nFiltered Shape:", df_filtered.shape)

# Optional sanity check on index range
print("\nIndex range:", df_filtered.index.min(), "to", df_filtered.index.max())

=== RAW DATA (first 5 rows) ===
    Name  Age      City
0  Alice   30  New York
1    Bob   25        LA
2  Carol   38   Chicago
0  Alice   30  New York
1    Bob   25        LA

Shape: (6, 3)
Columns: ['Name', 'Age', 'City']

=== FILTERED DATA (index > 100) ===
Empty DataFrame
Columns: [Name, Age, City]
Index: []

Filtered Shape: (0, 3)

Index range: nan to nan


## Working with Other Formats

### Stata Files

In [14]:
import pandas as pd

# Create a sample DataFrame
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol"],
    "age": [30, 25, 38],
    "city": ["New York", "Los Angeles", "Chicago"]
})

# Write to Stata format
df.to_stata(
    "data/output.dta",
    write_index=False  # avoids saving pandas index as a column
)

print("File written successfully!")

File written successfully!


In [15]:
import pandas as pd

# Read Stata file
df = pd.read_stata("data/output.dta")

# Check output
print(df.head())
print(df.shape)
print(df.columns)

    name  age         city
0  Alice   30     New York
1    Bob   25  Los Angeles
2  Carol   38      Chicago
(3, 3)
Index(['name', 'age', 'city'], dtype='object')


### Pickle Format

In [16]:
import pandas as pd

# Read pickled DataFrame
df = pd.read_pickle('data/data.pkl')

# Write to pickle format
df.to_pickle('data/output.pkl')

### Clipboard

### Clipboard

`pd.read_clipboard()` reads tabular data directly from your system clipboard, and `df.to_clipboard()` copies a DataFrame to it. These are useful for quick interactive work but **will not run inside a notebook** because there is no active clipboard session.

```python
import pandas as pd

# Read from system clipboard
df = pd.read_clipboard()

# Copy DataFrame to clipboard
df.to_clipboard(index=False)
```

## Data Type Preservation Across Formats

A critical issue when exporting data is that different formats preserve types differently. When you export to CSV, pandas converts everything to text — dates, categories, and numbers lose their type information on the round-trip.

In [18]:
import pandas as pd

# Create data with specific types
df = pd.DataFrame({
    'Date': pd.date_range('2024-01-01', periods=3),
    'Amount': [100.50, 200.75, 300.00],
    'Category': pd.Categorical(['A', 'B', 'A']),
    'Count': [10, 20, 30]
})

print("ORIGINAL TYPES:")
print(df.dtypes)

# Export to CSV and read back
df.to_csv('data.csv', index=False)
df_read = pd.read_csv('data.csv')

print("\nAFTER CSV ROUND-TRIP:")
print(df_read.dtypes)

ORIGINAL TYPES:
Date        datetime64[ns]
Amount             float64
Category          category
Count                int64
dtype: object

AFTER CSV ROUND-TRIP:
Date         object
Amount      float64
Category     object
Count         int64
dtype: object


**Output:**
```
ORIGINAL TYPES:
Date        datetime64[ns]
Amount             float64
Category          category
Count               int64

AFTER CSV ROUND-TRIP:
Date          object
Amount        float64
Category      object
Count         int64
```

**Solutions:**

In [20]:
import pandas as pd

# -----------------------------
# Sample DataFrame (example)
# -----------------------------
df = pd.DataFrame({
    "Date": pd.date_range("2024-01-01", periods=3),
    "Category": ["A", "B", "C"],
    "Value": [10, 20, 30]
})

# =========================================================
# Solution 1: CSV (manual type handling required)
# =========================================================

df.to_csv("data/data.csv", index=False)

df_csv = pd.read_csv(
    "data/data.csv",
    parse_dates=["Date"],                 # ensure Date is parsed correctly
    dtype={"Category": "category"}       # enforce categorical type
)

print("CSV result:")
print(df_csv.dtypes)
print(df_csv.head())


# =========================================================
# Solution 2: Parquet (BEST OPTION — preserves types)
# =========================================================

df.to_parquet("data/data.parquet", index=False)

df_parquet = pd.read_parquet("data/data.parquet")

print("\nParquet result:")
print(df_parquet.dtypes)
print(df_parquet.head())


# =========================================================
# Solution 3: Excel (good but not perfect type preservation)
# =========================================================

df.to_excel("data/data.xlsx", index=False)

df_excel = pd.read_excel("data/data.xlsx")

print("\nExcel result:")
print(df_excel.dtypes)
print(df_excel.head())

CSV result:
Date        datetime64[ns]
Category          category
Value                int64
dtype: object
        Date Category  Value
0 2024-01-01        A     10
1 2024-01-02        B     20
2 2024-01-03        C     30

Parquet result:
Date        datetime64[ns]
Category            object
Value                int64
dtype: object
        Date Category  Value
0 2024-01-01        A     10
1 2024-01-02        B     20
2 2024-01-03        C     30

Excel result:
Date        datetime64[ns]
Category            object
Value                int64
dtype: object
        Date Category  Value
0 2024-01-01        A     10
1 2024-01-02        B     20
2 2024-01-03        C     30


## Format Comparison Guide

| Format | Best For | Advantages | Disadvantages | Type Preservation |
|--------|----------|------------|---------------|-------------------|
| **CSV** | Data sharing, universal compatibility | Lightweight, opens anywhere | No formatting, loses type info | ⚠️ Limited |
| **Excel** | Business reports, presentations | Multiple sheets, styling | Larger files, requires openpyxl | ✓ Good |
| **JSON** | Web APIs, nested data | Supports complex structures | Larger than CSV | ⚠️ Limited |
| **Parquet** | Large datasets, analytics | Fast, compressed, columnar | Requires pyarrow | ✓ Excellent |
| **HDF5** | Scientific data, hierarchical | Fast I/O, filtering support | Requires tables library | ✓ Good |
| **SQL** | Database storage, queries | Structured, queryable | Requires DB setup | Varies |

**Quick Decision Guide:**

```
Are you sharing with non-technical users?
  → YES: Use Excel (.xlsx)
  → NO: Continue...

Is this for a web application or API?
  → YES: Use JSON (.json)
  → NO: Continue...

Is the file larger than 100 MB or do you need type preservation?
  → YES: Use Parquet (.parquet)
  → NO: Use CSV (.csv) ← safest default
```

## Practical Example: Multi-Format Data Pipeline

Here's a realistic example showing how to read data from multiple sources and consolidate them:

In [21]:
import pandas as pd
import sqlalchemy

# Read from different sources
csv_data = pd.read_csv('data/sales.csv')
excel_data = pd.read_excel('data/inventory.xlsx', sheet_name='Current')
json_data = pd.read_json('data/api_response.json', orient='records')

# Combine datasets
combined = pd.concat([csv_data, excel_data, json_data], ignore_index=True)

# Clean and transform
combined['date'] = pd.to_datetime(combined['date'])
combined = combined.dropna()

# Export to multiple formats
combined.to_csv('data/consolidated_data.csv', index=False)
combined.to_excel('data/consolidated_data.xlsx', index=False)
combined.to_json('data/consolidated_data.json', orient='records', date_format='iso')

# Store in database
engine = sqlalchemy.create_engine('sqlite:///data/analytics.db')
combined.to_sql('consolidated_data', engine, if_exists='replace', index=False)

7

## Performance Considerations

When working with large datasets, consider these optimization strategies:

In [22]:
import pandas as pd

# ----------------------------------------
# Example processing function (YOU define it)
# ----------------------------------------
def process_and_save(chunk, output_path="data/processed_output.csv"):
    """
    Processes a chunk of data and appends it to disk.
    """

    # Example processing (edit this for your use case)
    chunk["amount_scaled"] = chunk["amount"] * 0.1

    # Append to CSV (write header only once)
    chunk.to_csv(
        output_path,
        mode="a",
        index=False,
        header=not pd.io.common.file_exists(output_path)
    )


# ----------------------------------------
# Read CSV in chunks (memory efficient)
# ----------------------------------------
for chunk in pd.read_csv("data/huge_file.csv", chunksize=10000):

    # Optional: keep only needed columns
    chunk = chunk[["id", "amount", "date"]]

    # Optional: enforce dtypes (memory optimization)
    chunk["id"] = chunk["id"].astype("int32")
    chunk["amount"] = chunk["amount"].astype("float32")

    # Process + save each chunk
    process_and_save(chunk)


# ----------------------------------------
# Convert final result to Parquet (recommended)
# ----------------------------------------
df = pd.read_csv("data/processed_output.csv")
df.to_parquet("data/data.parquet", index=False)

print("Processing complete and saved to Parquet!")

Processing complete and saved to Parquet!


## Best Practices for Data I/O

1. **Always specify `index=False`** when writing to CSV or Excel unless your index contains meaningful data
2. **Use appropriate data types** — specify `dtype` when reading to save memory
3. **Handle encoding explicitly** — use `encoding='utf-8'` for non-ASCII characters
4. **Validate after import** — check shape, dtypes, and missing values
5. **Use compression** — add `compression='gzip'` to reduce CSV file size
6. **Choose the right format** — CSV for simplicity, Parquet for performance and type fidelity, JSON for APIs

In [25]:
import pandas as pd

# Sample data
df = pd.DataFrame({
    "id": [1, 2, 3],
    "price": [10.5, 20.0, 30.25],
    "name": ["A", "B", "C"]
})

# Save as gzip-compressed CSV
df.to_csv("data/data.csv.gz", index=False, compression="gzip")

print("Gzip file created successfully!")

# Read gzip-compressed CSV
df = pd.read_csv(
    "data/data.csv.gz",
    compression="gzip"
)

print(df.head())
print(df.info())

Gzip file created successfully!
   id  price name
0   1  10.50    A
1   2  20.00    B
2   3  30.25    C
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      3 non-null      int64  
 1   price   3 non-null      float64
 2   name    3 non-null      object 
dtypes: float64(1), int64(1), object(1)
memory usage: 200.0+ bytes
None


## Summary

Pandas provides a unified and intuitive API for data I/O across numerous formats. By mastering these tools, you can efficiently work with data regardless of its source or destination format. Choose the format that best suits your use case: CSV for simplicity and portability, Excel for business environments, JSON for web integration, SQL for relational databases, and binary formats like Parquet for performance-critical applications where type preservation matters.

---

# Communicating Results

Clear communication transforms your analysis into action. Whether you're writing a report, presenting to colleagues, or sharing findings online, pandas makes it easy to extract key insights and present them effectively.

## Exporting Results

### The `index` Parameter

The `index` parameter controls whether pandas saves row numbers to your file:

In [26]:
import pandas as pd

# Create sample data
college = pd.DataFrame({
    'Name': ['Harvard', 'MIT', 'Stanford'],
    'Location': ['Boston', 'Cambridge', 'Palo Alto'],
    'Founded': [1636, 1861, 1885]
})

# index=False: Don't save row numbers (recommended for most exports)
college.to_csv('college.csv', index=False)

# index=True: Save row numbers (use when index contains meaningful data)
college_by_year = college.set_index('Founded')
college_by_year.to_csv('college_indexed.csv', index=True)

**When to use `index=False`:** Most of the time — row numbers are usually just placeholders.

**When to use `index=True`:** When your index contains meaningful information (dates, IDs, custom labels).

### Basic Export Examples

In [27]:
import pandas as pd

college = pd.read_csv('college.csv')

# Simple CSV export
college.to_csv('cleaned_college.csv', index=False)
print("✓ Saved to cleaned_college.csv")

# Export with specific columns only
college[['Name', 'Location']].to_csv('college_locations.csv', index=False)

# Export with custom encoding (handles special characters)
college.to_csv('college_utf8.csv', index=False, encoding='utf-8')

✓ Saved to cleaned_college.csv


### Formatting for Your Audience

Raw numbers are hard to read. Format exports to make them presentation-ready:

In [28]:
import pandas as pd

college = pd.DataFrame({
    'TUITION': [12500.50, 15000.00],
    'ENROLLMENT': [5000.00, 8500.00],
    'SAT_AVG': [1200.75, 1350.25]
})

# Control decimal places in CSV export
college.to_csv(
    'cleaned_college.csv',
    index=False,
    float_format='%.2f'
)

**Result:**
```
TUITION,ENROLLMENT,SAT_AVG
12500.50,5000.00,1200.75
15000.00,8500.00,1350.25
```

### Common Export Pitfalls

In [29]:
import pandas as pd

college = pd.read_csv('college.csv')

# ❌ PITFALL 1: Index bleeding into data
college.to_csv('bad_export.csv')  # Includes unwanted index column!

# ✓ FIX: Always use index=False for clean exports
college.to_csv('good_export.csv', index=False)

# ❌ PITFALL 2: Encoding errors with special characters
# college.to_csv('bad_encoding.csv', encoding='ascii')  # Fails on non-ASCII

# ✓ FIX: Use UTF-8 encoding
college.to_csv('good_encoding.csv', encoding='utf-8')

# ❌ PITFALL 3: Losing precision on large numbers
df = pd.DataFrame({'BigNumber': [123456789.123456789]})
df.to_csv('precision_loss.csv')  # May round unexpectedly

# ✓ FIX: Specify precision explicitly
df.to_csv('precision_kept.csv', float_format='%.10f')

---

## Creating Summary Tables

### Basic Grouping and Aggregation

In [31]:
college.head()

,INSTNM,CITY,STABBR,HBCU,MENONLY,WOMENONLY,RELAFFIL,SATVRMID,SATMTMID,DISTANCEONLY,...,UGDS_2MOR,UGDS_NRA,UGDS_UNKN,PPTUG_EF,CURROPER,PCTPELL,PCTFLOAN,UG25ABV,MD_EARN_WNE_P10,GRAD_DEBT_MDN_SUPP
0,Alabama A & M University,Normal,AL,1.0,0.0,0.0,0,424.0,420.0,0.0,...,0.0000,0.0059,0.0138,0.0656,1,0.7356,0.8284,0.1049,30300,33888
1,University of Alabama at Birmingham,Birmingham,AL,0.0,0.0,0.0,0,570.0,565.0,0.0,...,0.0368,0.0179,0.0100,0.2607,1,0.3460,0.5214,0.2422,39700,21941.5
2,Amridge University,Montgomery,AL,0.0,0.0,0.0,1,NaN,NaN,1.0,...,0.0000,0.0000,0.2715,0.4536,1,0.6801,0.7795,0.8540,40100,23370
3,University of Alabama in Huntsville,Huntsville,AL,0.0,0.0,0.0,0,595.0,590.0,0.0,...,0.0172,0.0332,0.0350,0.2146,1,0.3072,0.4596,0.2640,45500,24097
4,Alabama State University,Montgomery,AL,1.0,0.0,0.0,0,425.0,430.0,0.0,...,0.0098,0.0243,0.0137,0.0892,1,0.7347,0.7554,0.1270,26600,33118.5
